# AURORA CORE — Colab GPU Worker + Ollama Runtime

Connect a Google Colab GPU runtime to your AURORA CORE backend and run model inference via Ollama.

**Steps:**
1. Runtime → Change runtime type → **GPU** (T4 recommended)
2. Run Cell 1 to configure
3. Run Cell 2 to detect GPU
4. Run Cell 3 to connect worker + start heartbeat + job polling
5. Run Cell 4 to run GPU benchmark
6. Run Cell 5 to install dependencies + Ollama
7. Run Cell 6 to verify Ollama + approved model
8. Run Cell 7 to verify GPU execution + run live inference
9. Run Cell 8 to check runtime status
10. Run Cell 9 to disconnect cleanly

**GPU Verification:** Cell 7 checks nvidia-smi, ollama list, ollama ps, runs inference,
then checks ollama ps again to confirm GPU vs CPU execution.

**AURORA does NOT:**
- Access your Google account
- Automate login
- Store your credentials
- Accept arbitrary code execution
- Expose Ollama to the internet

In [ ]:
#@title Cell 1: Configure Connection { display-mode: "form" }
#@markdown Enter your AURORA backend URL and worker token.
#@markdown The token is entered securely (not stored in notebook).

import getpass
import os

#@markdown ---
#@markdown **AURORA Backend URL** (production Render backend):
AURORA_BACKEND_URL = "https://aurora-core-1-txvl.onrender.com" #@param {type:"string"}
#@markdown ---

print(f"Backend URL: {AURORA_BACKEND_URL}")
print("\nEnter your AURORA worker token (input will be hidden):")
AURORA_WORKER_TOKEN = getpass.getpass("Worker token: ")

if not AURORA_WORKER_TOKEN:
    raise ValueError("Worker token cannot be empty")

print("\nConfiguration saved. Token is masked and not stored.")
print(f"Backend: {AURORA_BACKEND_URL}")
print(f"Token: {'*' * 8}{AURORA_WORKER_TOKEN[-4:] if len(AURORA_WORKER_TOKEN) > 4 else '****'}")

In [ ]:
#@title Cell 2: Detect GPU { display-mode: "form" }
#@markdown Detects GPU hardware from the Colab runtime.
#@markdown Only reports values actually detected by PyTorch CUDA or nvidia-smi.

GPU_INFO = {
    "name": "UNKNOWN",
    "vendor": "UNKNOWN",
    "vram_mb": 0.0,
    "cuda_version": None,
    "driver_version": None,
    "compute_capability": None,
    "available_memory_mb": 0.0,
    "runtime_info": None,
}

try:
    import torch
    if torch.cuda.is_available():
        GPU_INFO["name"] = torch.cuda.get_device_name(0)
        GPU_INFO["vendor"] = "NVIDIA"
        GPU_INFO["cuda_version"] = torch.version.cuda
        GPU_INFO["compute_capability"] = ".".join(str(x) for x in torch.cuda.get_device_capability(0))
        mem = torch.cuda.get_device_properties(0)
        GPU_INFO["vram_mb"] = round(mem.total_mem / (1024 * 1024), 1)
        GPU_INFO["available_memory_mb"] = round(
            (mem.total_mem - torch.cuda.memory_allocated(0)) / (1024 * 1024), 1
        )
        GPU_INFO["runtime_info"] = f"PyTorch {torch.__version__}"
        print(f"GPU: {GPU_INFO['name']}")
        print(f"VRAM: {GPU_INFO['vram_mb']} MB")
        print(f"CUDA: {GPU_INFO['cuda_version']}")
        print(f"Compute Capability: {GPU_INFO['compute_capability']}")
        print(f"Available Memory: {GPU_INFO['available_memory_mb']} MB")
        print(f"PyTorch: {torch.__version__}")
    else:
        print("WARNING: CUDA not available. GPU inference will not work.")
except ImportError:
    print("WARNING: PyTorch not installed. Run: pip install torch")

print(f"\nGPU Info: {GPU_INFO['name']} ({GPU_INFO['vram_mb']} MB)")

In [ ]:
#@title Cell 3: Connect Worker { display-mode: "form" }
#@markdown Connects to AURORA backend, starts heartbeat and job polling.

import sys
import os
import time

# Add worker to path
WORKER_DIR = "/content/aurora-core/workers/google_colab"
if not os.path.exists(WORKER_DIR):
    print("Cloning AURORA CORE repository...")
    !git clone https://github.com/999shotff/aurora-core.git /content/aurora-core 2>/dev/null || true

if WORKER_DIR not in sys.path:
    sys.path.insert(0, WORKER_DIR)

from worker import AuroraColabWorker

worker = AuroraColabWorker(
    backend_url=AURORA_BACKEND_URL,
    worker_token=AURORA_WORKER_TOKEN,
)

if worker.connect():
    print(f"Worker connected: {worker.worker_id}")
    print(f"GPU: {worker._gpu_info.get('name', 'UNKNOWN')}")
    print("Heartbeat and job polling started.")
    print("\nWorker is READY to accept jobs.")
else:
    print("ERROR: Failed to connect to backend.")
    print("Check your backend URL and worker token.")

In [ ]:
#@title Cell 4: GPU Benchmark { display-mode: "form" }
#@markdown Runs a real GPU matrix multiply benchmark.

import hashlib
import time

try:
    import torch
    if not torch.cuda.is_available():
        print("ERROR: CUDA not available. Cannot run benchmark.")
    else:
        matrix_size = 1024
        iterations = 10
        print(f"Running benchmark: {matrix_size}x{matrix_size} matrix, {iterations} iterations")
        print(f"GPU: {torch.cuda.get_device_name(0)}")

        start = time.time()
        a = torch.randn(matrix_size, matrix_size, device="cuda")
        b = torch.randn(matrix_size, matrix_size, device="cuda")
        for _ in range(iterations):
            _ = torch.mm(a, b)
        torch.cuda.synchronize()
        elapsed = time.time() - start
        total_flops = 2.0 * matrix_size ** 3 * iterations
        gflops = total_flops / elapsed / 1e9
        checksum = hashlib.sha256(f"gpu-{matrix_size}-{iterations}".encode()).hexdigest()[:16]

        print(f"\nBenchmark PASSED")
        print(f"Time: {elapsed:.3f}s")
        print(f"GFLOPS: {gflops:.2f}")
        print(f"Checksum: {checksum}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"Benchmark FAILED: {e}")

In [ ]:
#@title Cell 5: Install Ollama + CUDA GPU Configuration { display-mode: "form" }
#@markdown Installs zstd (required by Ollama installer), then Ollama with CUDA GPU configuration.
#@markdown Ollama runs locally on the Colab machine (localhost:11434).
#@markdown It is NOT exposed to the internet.

import subprocess
import time
import shutil
import requests
import os
import glob

OLLAMA_URL = "http://127.0.0.1:11434"

# ── Step 1: Configure CUDA environment ───────────────────────────
print("[1] Configuring CUDA environment...")
cuda_lib_paths = [
    "/usr/local/cuda/lib64",
    "/usr/lib/x86_64-linux-gnu",
]
for p in glob.glob("/usr/local/cuda-*/lib64"):
    cuda_lib_paths.append(p)

existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
ld_path_str = ":".join(cuda_lib_paths)
if existing_ld:
    ld_path_str = ld_path_str + ":" + existing_ld
os.environ["LD_LIBRARY_PATH"] = ld_path_str
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"    LD_LIBRARY_PATH={ld_path_str}")
print(f"    CUDA_VISIBLE_DEVICES=0")

# ── Step 2: Verify Ollama binary exists ────────────────────────────
print("\n[2] Checking Ollama binary...")
ollama_bin = shutil.which("ollama")
if not ollama_bin:
    # Install prerequisites and Ollama
    print("    Ollama not found. Installing prerequisites...")
    zstd_path = shutil.which("zstd")
    if not zstd_path:
        print("    Installing zstd...")
        !apt-get update -qq && apt-get install -y -qq zstd
        if not shutil.which("zstd"):
            raise RuntimeError("Failed to install zstd.")
        print("    zstd installed.")
    else:
        print(f"    zstd found: {zstd_path}")

    print("    Installing Ollama...")
    install_result = !curl -fsSL https://ollama.com/install.sh | sh 2>&1
    for line in install_result:
        print(f"    {line}")
    if any("error" in line.lower() or "failed" in line.lower() for line in install_result):
        raise RuntimeError(f"Ollama installation failed.")
    ollama_bin = shutil.which("ollama")
    if not ollama_bin:
        raise RuntimeError("Ollama binary not found after installation.")

print(f"    Binary: {ollama_bin}")

# ── Step 3: Kill any existing Ollama process ──────────────────────
print("\n[3] Cleaning up any existing Ollama process...")
!pkill -f 'ollama serve' 2>/dev/null || true
time.sleep(1)

# ── Step 4: Start Ollama service with CUDA environment ────────────
print("\n[4] Starting Ollama service with CUDA environment...")
!nohup env LD_LIBRARY_PATH={ld_path_str} CUDA_VISIBLE_DEVICES=0 ollama serve > /tmp/ollama.log 2>&1 &

# ── Step 5: Wait for Ollama health (port polling) ─────────────────
print("\n[5] Waiting for Ollama to become healthy...")
ollama_running = False
for attempt in range(15):
    try:
        resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
        if resp.status_code == 200:
            ollama_running = True
            print(f"    Ollama responded after {attempt + 1} attempt(s).")
            break
    except Exception:
        pass
    time.sleep(1)

if not ollama_running:
    log_content = ""
    try:
        with open("/tmp/ollama.log") as f:
            log_content = f.read()[-500:]
    except Exception:
        log_content = "(could not read log)"
    raise RuntimeError(
        f"Ollama failed to start after 15s. Log:\n{log_content}"
    )

# ── Step 6: Verify Ollama version and API ─────────────────────────
print("\n[6] Verifying Ollama...")
version_check = !ollama --version 2>&1
print(f"    Version: {version_check[0] if version_check else 'unknown'}")
try:
    resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    models = resp.json().get("models", [])
    print(f"    API status: {resp.status_code}")
    print(f"    Available models: {len(models)}")
    for m in models:
        print(f"      - {m.get('name', 'unknown')}")
except Exception as e:
    raise RuntimeError(f"Ollama API verification failed: {e}")

# ── Step 7: Check Ollama log for GPU/CUDA information ─────────────
print("\n[7] Checking Ollama log for GPU/CUDA information...")
try:
    with open("/tmp/ollama.log") as f:
        log_content = f.read()
    cuda_lines = [line for line in log_content.split("\n")
                  if any(kw in line.lower() for kw in ["cuda", "gpu", "nvidia", "device", "acceler"])]
    if cuda_lines:
        print("    GPU-related log entries:")
        for line in cuda_lines:
            print(f"      {line}")
    else:
        print("    No GPU/CUDA lines found in log.")
        print("    Ollama may be running on CPU only.")
except FileNotFoundError:
    print("    Log file not found.")

print("\n=== Ollama Setup: COMPLETE ===")

In [ ]:
#@title Cell 6: Verify Ollama + Approved Model { display-mode: "form" }
#@markdown Pulls the approved Ollama model if not present.
#@markdown Only the approved model may be used.
#@markdown Verifies GPU usage via ollama ps after model load.

import requests
import time

OLLAMA_URL = "http://127.0.0.1:11434"
APPROVED_MODEL = "qwen2.5:0.5b"

# Check Ollama health
try:
    resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    if resp.status_code != 200:
        print("ERROR: Ollama is not responding.")
    else:
        models = [m.get("name", "") for m in resp.json().get("models", [])]
        print(f"Ollama: READY")
        print(f"Available models: {models}")

        if APPROVED_MODEL in models:
            print(f"\nApproved model '{APPROVED_MODEL}' is available.")
        else:
            print(f"\nPulling approved model: {APPROVED_MODEL}...")
            pull_resp = requests.post(f"{OLLAMA_URL}/api/pull",
                                     json={"name": APPROVED_MODEL},
                                     timeout=300)
            if pull_resp.status_code == 200:
                print(f"Model '{APPROVED_MODEL}' pulled successfully.")
            else:
                print(f"ERROR: Failed to pull model: {pull_resp.text[:200]}")

        # Verify model exists
        resp2 = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        models2 = [m.get("name", "") for m in resp2.json().get("models", [])]
        if APPROVED_MODEL in models2:
            print(f"\nModel verification: PASSED")
            print(f"Model: {APPROVED_MODEL}")
        else:
            print(f"\nModel verification: FAILED")
            print(f"'{APPROVED_MODEL}' not found after pull.")
except Exception as e:
    print(f"ERROR: {e}")

In [ ]:
#@title Cell 7: GPU Verification + Live Inference { display-mode: "form" }
#@markdown Runs inference through Ollama and verifies GPU execution.
#@markdown The prompt is bounded and safe.

import hashlib
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
APPROVED_MODEL = "qwen2.5:0.5b"

PROMPT = "Explain in two short sentences why evidence and uncertainty matter in scientific analysis."
MAX_TOKENS = 256
TEMPERATURE = 0.7

print("=" * 60)
print("GPU VERIFICATION SEQUENCE")
print("=" * 60)

# ── A. Check nvidia-smi BEFORE inference ────────────────────────────
print("\n[A] nvidia-smi (before inference):")
try:
    smi_before = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=5,
    )
    if smi_before.returncode == 0:
        print(f"    {smi_before.stdout.strip()}")
    else:
        print("    nvidia-smi not available")
except FileNotFoundError:
    print("    nvidia-smi not found")

# ── B. Check ollama list ────────────────────────────────────────────
print("\n[B] ollama list:")
try:
    ollama_list = subprocess.run(
        ["ollama", "list"], capture_output=True, text=True, timeout=10,
    )
    if ollama_list.returncode == 0:
        for line in ollama_list.stdout.strip().split("\n"):
            print(f"    {line}")
    else:
        print(f"    {ollama_list.stderr.strip()}")
except FileNotFoundError:
    print("    ollama CLI not found")

# ── C. Check ollama ps (no models loaded yet) ──────────────────────
print("\n[C] ollama ps (before inference):")
try:
    ollama_ps = subprocess.run(
        ["ollama", "ps"], capture_output=True, text=True, timeout=10,
    )
    if ollama_ps.returncode == 0:
        for line in ollama_ps.stdout.strip().split("\n"):
            print(f"    {line}")
    else:
        print(f"    {ollama_ps.stderr.strip()}")
except FileNotFoundError:
    print("    ollama CLI not found")

# ── D. Run actual inference ─────────────────────────────────────────
print(f"\n[D] Running inference on {APPROVED_MODEL}...")
print(f"    Prompt: {PROMPT[:60]}...")
start = time.time()
gpu_detected_during = False
output = ""
tokens = 0
error_msg = None
try:
    result = requests.post(f"{OLLAMA_URL}/api/generate", json={
        "model": APPROVED_MODEL,
        "prompt": PROMPT,
        "options": {
            "num_predict": MAX_TOKENS,
            "temperature": TEMPERATURE,
        },
        "stream": False,
    }, timeout=120)
    elapsed = time.time() - start

    if result.status_code == 200:
        data = result.json()
        output = data.get("response", "")
        tokens = data.get("eval_count", len(output.split()))
        eval_duration_ns = data.get("eval_duration", 0)
        eval_duration_s = eval_duration_ns / 1e9 if eval_duration_ns else elapsed
    else:
        error_msg = f"Ollama returned {result.status_code}"
        elapsed = time.time() - start
except Exception as e:
    elapsed = time.time() - start
    error_msg = str(e)

# ── E. Check ollama ps DURING/AFTER inference ──────────────────────
print("\n[E] ollama ps (after inference):")
try:
    ollama_ps2 = subprocess.run(
        ["ollama", "ps"], capture_output=True, text=True, timeout=10,
    )
    if ollama_ps2.returncode == 0:
        ps_output = ollama_ps2.stdout.strip()
        for line in ps_output.split("\n"):
            print(f"    {line}")
        # Check processor column for GPU/CPU
        if "gpu" in ps_output.lower() or "cuda" in ps_output.lower():
            gpu_detected_during = True
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

# ── F. Check nvidia-smi DURING/AFTER inference ─────────────────────
print("\n[F] nvidia-smi (after inference):")
try:
    smi_after = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=5,
    )
    if smi_after.returncode == 0:
        print(f"    {smi_after.stdout.strip()}")
    else:
        print("    nvidia-smi not available")
except FileNotFoundError:
    print("    nvidia-smi not found")

# ── G. Determine GPU/CPU execution ─────────────────────────────────
print("\n[G] Execution status:")
gpu_available = GPU_INFO.get("name", "UNKNOWN") != "UNKNOWN"
if gpu_detected_during:
    execution_status = "GPU_ACCELERATED"
    print(f"    Model: {APPROVED_MODEL}")
    print(f"    Processor: GPU (confirmed via ollama ps)")
elif gpu_available:
    execution_status = "GPU_AVAILABLE_BUT_NOT_USED"
    print(f"    Model: {APPROVED_MODEL}")
    print(f"    Processor: CPU (GPU available but not used)")
else:
    execution_status = "CPU_ONLY"
    print(f"    Model: {APPROVED_MODEL}")
    print(f"    Processor: CPU (no GPU detected)")

# ── Report ─────────────────────────────────────────────────────────
input_hash = hashlib.sha256(PROMPT.encode()).hexdigest()[:16]
output_hash = hashlib.sha256(output.encode()).hexdigest()[:16]

print(f"\n{'=' * 60}")
print("LIVE INFERENCE RESULT")
print(f"{'=' * 60}")
print(f"Model: {APPROVED_MODEL}")
print(f"Runtime: Ollama (local)")
print(f"Duration: {elapsed:.3f}s")
print(f"Tokens: {tokens}")
print(f"Tokens/sec: {tokens / elapsed:.1f}" if elapsed > 0 else "N/A")
print(f"Input hash: {input_hash}")
print(f"Output hash: {output_hash}")
print(f"GPU status: {execution_status}")
print(f"\n--- Output ---")
print(output)
print(f"{'=' * 60}")
if error_msg:
    print(f"\nLIVE MODEL INFERENCE VERIFIED: NO (error: {error_msg})")
else:
    print(f"\nLIVE MODEL INFERENCE VERIFIED: YES")

In [ ]:
#@title Cell 8: Runtime Status { display-mode: "form" }
#@markdown Shows current runtime status including Ollama GPU/CPU classification.

import requests

OLLAMA_URL = "http://127.0.0.1:11434"

print("=== AURORA Runtime Status ===")
print(f"Worker: {worker.worker_id if worker else 'NOT CONNECTED'}")
print(f"GPU: {worker._gpu_info.get('name', 'UNKNOWN') if worker else 'UNKNOWN'}")
print(f"VRAM: {worker._gpu_info.get('vram_mb', 0)} MB" if worker else "N/A")

# Ollama status
try:
    resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    if resp.status_code == 200:
        models = resp.json().get("models", [])
        print(f"\nOllama: READY")
        print(f"Ollama URL: {OLLAMA_URL} (local)")
        print(f"Available models: {len(models)}")
        for m in models:
            print(f"  - {m.get('name', 'unknown')}")
    else:
        print(f"\nOllama: ERROR (status {resp.status_code})")
except Exception:
    print(f"\nOllama: NOT REACHABLE")

# Runtime handler status
if worker and worker._runtime_handler:
    rh = worker._runtime_handler
    print(f"\nTransformers Runtime: {'LOADED' if rh.is_model_loaded else 'NOT LOADED'}")
    if rh.is_model_loaded:
        print(f"  Model: {rh.loaded_model_id}")

if worker and worker._ollama_runtime:
    orh = worker._ollama_runtime
    print(f"\nOllama Runtime: {'LOADED' if orh.is_model_loaded else 'NOT LOADED'}")
    if orh.is_model_loaded:
        print(f"  Model: {orh.loaded_model_id}")
    print(f"  GPU status: {orh._last_gpu_status}")
    print(f"  Total inferences: {orh._total_inferences}")
    print(f"  Total errors: {orh._total_errors}")

In [ ]:
#@title Cell 9: Disconnect { display-mode: "form" }
#@markdown Gracefully disconnects the worker from the AURORA backend.

if worker:
    worker.disconnect()
    print("Worker disconnected from AURORA backend.")
else:
    print("No worker to disconnect.")